# Personal Finance Dataset

In [2]:
import pandas as pd
import numpy as np
import os
import re

Load Dataset

In [3]:
df = pd.read_csv("../Datasets/Personal_Finance_Dataset.csv")
print("[INFO] Loaded:", df.shape)

[INFO] Loaded: (1500, 5)


Remove Duplicates

In [4]:
df = df.drop_duplicates()
print("[INFO] After removing duplicates:", df.shape)

[INFO] After removing duplicates: (1500, 5)


Standardize Schema

In [5]:
df = df.rename(columns={
    "Description": "Transaction Description",
    "desc": "Transaction Description",
    "category": "Category",
    "date": "Date",
    "amount": "Amount",
    "type": "Type"
})
print("[INFO] Columns:", df.columns.tolist())

[INFO] Columns: ['Date', 'Transaction Description', 'Category', 'Amount', 'Type']


Fix Date Column

In [6]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"])
print("[INFO] Valid dates:", df["Date"].notna().sum())

[INFO] Valid dates: 1500


Fix Amount column

In [7]:
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
df = df[df["Amount"] > 0]
print("[INFO] Valid amounts:", df["Amount"].notna().sum())

[INFO] Valid amounts: 1500


Fix missing text fields

In [8]:
df["Transaction Description"] = df["Transaction Description"].fillna("").astype(str)
df["Category"] = df["Category"].fillna("Other").astype(str)

FIX & IMPUTE TYPE

In [9]:
# Normalize
df["Type"] = df["Type"].astype(str).str.strip().str.title()

# Correct salary being marked as Expense → fix this!
df.loc[df["Category"].str.lower() == "salary", "Type"] = "Income"

# Category-based income inference
income_like = [
    "salary", "bonus", "interest income", "investment income",
    "dividend", "refund", "other income", "tax refund", "side income"
]

mask_income = (
    df["Type"].isna() &
    df["Category"].str.lower().isin(income_like)
)

df.loc[mask_income, "Type"] = "Income"

# Remaining missing → assign majority class (Expense)
df["Type"] = df["Type"].fillna("Expense")

print("[INFO] Type distribution:")
print(df["Type"].value_counts())

[INFO] Type distribution:
Type
Expense    1076
Income      424
Name: count, dtype: int64


Add UserID

In [10]:
df["UserID"] = "US001"

Add TransactionID

In [11]:
df["TransactionID"] = [f"PFD{idx:06d}" for idx in range(1, len(df) + 1)]

Add Missing Optional Columns

In [12]:
for col in ["Currency", "Merchant", "Account Name"]:
    if col not in df.columns:
        df[col] = ""

Currency Inference

In [13]:
# Single cell: detect rows with currency hints (no new columns), then show test examples
import re, pandas as pd
from IPython.display import display

# CHANGE if your description column name differs
desc_col = 'Transaction Description'

# --- strict maps (no ambiguous short tokens like 'US' or 'IN') ---
SYMBOLS = {'$','£','€','₹','¥','Rs','₨'}
CODES = ['USD','GBP','EUR','INR','LKR','AUD','CAD','NZD','SGD','JPY','CNY','HKD','PKR','ZAR','AED','SAR']
COUNTRY_KEYWORDS = [
    'UNITED STATES OF AMERICA','UNITED STATES','USA',
    'UNITED KINGDOM','UK',
    'COLOMBO','.LK','SRI LANKA','LKR',
    'INDIA','INR',
    'AUSTRALIA','AUD','CANADA','CAD',
    'SINGAPORE','SGD','EURO','EUR','JAPAN','JPY','CHINA','CNY','HONG KONG','HKD'
]

# compile patterns
symbol_patterns = []
for s in SYMBOLS:
    if re.search(r'[A-Za-z]', s):
        symbol_patterns.append(re.compile(r'(?<!\w)'+re.escape(s)+r'\.?(?!\w)', flags=re.IGNORECASE))
    else:
        symbol_patterns.append(re.compile(re.escape(s)))
code_pat = re.compile(r'\b(?:' + '|'.join([re.escape(c) for c in CODES]) + r')\b', flags=re.IGNORECASE)
kw_parts = []
for kw in COUNTRY_KEYWORDS:
    if re.search(r'\W', kw):
        kw_parts.append(re.escape(kw))
    else:
        kw_parts.append(r'\b' + re.escape(kw) + r'\b')
country_pat = re.compile('|'.join(sorted(kw_parts, key=len, reverse=True)), flags=re.IGNORECASE)

def _has_hint(text):
    if pd.isna(text):
        return False
    s = str(text)
    # symbols
    for pat in symbol_patterns:
        if pat.search(s):
            return True
    # explicit 3-letter codes
    if code_pat.search(s):
        return True
    # country/domain keywords
    if country_pat.search(s):
        return True
    return False

# --- get boolean mask and matched rows (no modification to df) ---
mask = df[desc_col].astype(str).apply(_has_hint)
matched = df[mask]

print(f"Found {len(matched)} matching rows out of {len(df)} total. Showing up to first 200 matches:\n")
display(matched.head(200))

Found 0 matching rows out of 1500 total. Showing up to first 200 matches:



,Date,Transaction Description,Category,Amount,Type,UserID,TransactionID,Currency,Merchant,Account Name


In [14]:
# groupby income and expense
income = df[df['Type'] == 'Income']
expense = df[df['Type'] == 'Expense']

print(f"Total Income Transactions: {len(income)}")
print(f"Total Expense Transactions: {len(expense)}")

Total Income Transactions: 424
Total Expense Transactions: 1076


In [15]:
# Copy to avoid damaging original
df_copy = df.copy()

# Convert Date column to datetime (replace 'Date' with your actual column name)
df_copy["Date"] = pd.to_datetime(df_copy["Date"])

# Extract year-month
df_copy["YearMonth"] = df_copy["Date"].dt.to_period("M")

# Split income and expense
income = df_copy[df_copy["Type"] == "Income"]
expense = df_copy[df_copy["Type"] == "Expense"]

# Group by month and sum
monthly_income = income.groupby("YearMonth")["Amount"].sum()
monthly_expense = expense.groupby("YearMonth")["Amount"].sum()

# Average per month
avg_monthly_income = monthly_income.mean()
avg_monthly_expense = monthly_expense.mean()

print(f"Average Monthly Income: {avg_monthly_income:.2f}")
print(f"Average Monthly Expense: {avg_monthly_expense:.2f}")

Average Monthly Income: 14719.01
Average Monthly Expense: 17969.01


In [16]:
# Filter only salary transactions (case insensitive search in Category)
salary = df_copy[df_copy["Category"].str.contains("salary", case=False, na=False)]

# Sort by date
salary = salary.sort_values("Date")

print("Sample Salary Transactions:")
print(salary.head())

# Check time gaps between salary payments
salary["Gap_days"] = salary["Date"].diff().dt.days
print("\nGaps between salary transactions (in days):")
print(salary["Gap_days"].value_counts().sort_index())

# Detect pattern
avg_gap = salary["Gap_days"].mean()
print(f"\nAverage gap between salary transactions: {avg_gap:.1f} days")

if 25 <= avg_gap <= 35:
    print("➡ Likely a MONTHLY salary")
elif 13 <= avg_gap <= 15:
    print("➡ Likely a BI-WEEKLY salary")
elif 6 <= avg_gap <= 8:
    print("➡ Likely a WEEKLY salary")
elif avg_gap <= 2:
    print("➡ Likely a DAILY wage")
else:
    print("➡ Irregular salary pattern")

# Average salary per month
salary["YearMonth"] = salary["Date"].dt.to_period("M")
monthly_salary = salary.groupby("YearMonth")["Amount"].sum()

avg_monthly_salary = monthly_salary.mean()
print(f"\nAverage Monthly Salary: {avg_monthly_salary:.2f}")

Sample Salary Transactions:
         Date        Transaction Description Category   Amount    Type UserID  \
11 2020-01-26       Range successful simply.   Salary  1077.09  Income  US001   
27 2020-02-13  Building different full open.   Salary   294.31  Income  US001   
58 2020-04-06                    Away third.   Salary   493.56  Income  US001   
61 2020-04-17               Government nice.   Salary  1729.04  Income  US001   
74 2020-05-03          I fast camera inside.   Salary   318.05  Income  US001   

   TransactionID Currency Merchant Account Name YearMonth  
11     PFD000012                                  2020-01  
27     PFD000028                                  2020-02  
58     PFD000059                                  2020-04  
61     PFD000062                                  2020-04  
74     PFD000075                                  2020-05  

Gaps between salary transactions (in days):
Gap_days
0.0      5
1.0     14
2.0      4
3.0      9
4.0     11
5.0      8
6.0  

In [17]:
# --- Step 1: Amount Statistics ---
amount_stats = df['Amount'].describe()
print("Transaction Amount Statistics:")
print(amount_stats)
print("\n")

# --- Step 2: Check amount range logic ---
median_amt = amount_stats['50%']
if 10 <= median_amt <= 5000:
    print(f"Median amount ~{median_amt:.2f} fits typical USD personal transactions (groceries, bills, mortgage).")
else:
    print(f"Median amount ~{median_amt:.2f} is unusual for USD — may suggest another currency.")
print("\n")

# --- Step 3: Merchant Evidence ---
us_merchants = {
    "target", "walmart", "starbucks", "amazon", "netflix", "best buy",
    "state farm", "quiktrip", "chick-fil-a", "bojangles", "sheetz",
    "chevron", "bp", "conoco", "valero"
}

# Normalize merchant column if available
merchant_col = None
for col in df.columns:
    if col.lower() in ["merchant", "description"]:
        merchant_col = col
        break

if merchant_col:
    merchants_found = set(df[merchant_col].str.lower().unique())
    us_matches = merchants_found.intersection(us_merchants)
    
    print("Merchant Evidence:")
    print(f"U.S. brands detected in dataset → {us_matches}")
    
    if len(us_matches) > 3:
        print("Strong evidence dataset is from the U.S. context → Currency = USD ($)")
    else:
        print("Few U.S. brands detected. Currency inference uncertain.")
else:
    print("No Merchant/Description column found for brand matching.")

Transaction Amount Statistics:
count    1500.000000
mean     1307.520913
std       982.283361
min        14.370000
25%       629.340000
50%      1156.285000
75%      1712.932500
max      4996.000000
Name: Amount, dtype: float64


Median amount ~1156.28 fits typical USD personal transactions (groceries, bills, mortgage).


Merchant Evidence:
U.S. brands detected in dataset → set()
Few U.S. brands detected. Currency inference uncertain.


In [18]:
# Step 1: Amount stats
amount_stats = df['Amount'].describe()
median_amt = amount_stats['50%']

print("Amount Statistics:")
print(amount_stats, "\n")

if 10 <= median_amt <= 6000:
    print(f"Median amount ~{median_amt:.2f} fits typical USD household spending patterns.")
else:
    print(f"Median amount ~{median_amt:.2f} does not fit USD ranges.\n")

# Step 2: Category evidence
us_categories = {"rent", "utilities", "food & drink", "investment", "mortgage payment", "biweekly paycheck"}
categories_found = set(df['Category'].str.lower().unique())

print("Category Evidence:")
print(f"Categories in dataset: {categories_found}")

if categories_found.intersection(us_categories):
    print("Dataset uses U.S.-style categories (Rent, Utilities, Biweekly Paycheck, Mortgage). Strong evidence of USD context.")
else:
    print("Categories do not strongly indicate USD.")

# Final Conclusion
print("\nConclusion: Based on transaction amounts and category naming, currency is most likely USD ($).")

Amount Statistics:
count    1500.000000
mean     1307.520913
std       982.283361
min        14.370000
25%       629.340000
50%      1156.285000
75%      1712.932500
max      4996.000000
Name: Amount, dtype: float64 

Median amount ~1156.28 fits typical USD household spending patterns.
Category Evidence:
Categories in dataset: {'other', 'investment', 'rent', 'entertainment', 'health & fitness', 'salary', 'travel', 'shopping', 'utilities', 'food & drink'}
Dataset uses U.S.-style categories (Rent, Utilities, Biweekly Paycheck, Mortgage). Strong evidence of USD context.

Conclusion: Based on transaction amounts and category naming, currency is most likely USD ($).


In [19]:
# getting USD as Currency 
df['Currency'] = "USD"

Merchant Assignment

In [20]:
# creating Merchatn baed on description and Category 
import numpy as np

# Define merchant mapping
merchant_map = {
    "food & drink": ["McDonalds", "Starbucks", "Subway", "Dunkin"],
    "utilities": ["Verizon", "AT&T", "Duke Energy", "Con Edison"],
    "rent": ["Landlord", "Zillow Rentals", "Apartment Complex"],
    "investment": ["Vanguard", "Fidelity", "Charles Schwab", "Robinhood"],
    "entertainment": ["Netflix", "Spotify", "AMC Theatres", "Hulu"],
    "transportation": ["Uber", "Lyft", "Shell", "Chevron", "Exxon"],
    "healthcare": ["CVS Pharmacy", "Walgreens", "Local Clinic"],
    "shopping": ["Walmart", "Target", "Amazon", "Best Buy"]
}

# Function to assign merchant based on category
def assign_merchant(category):
    category = category.lower().strip()
    if category in merchant_map:
        return np.random.choice(merchant_map[category])
    return "Generic Merchant"

# Apply function
df['Merchant'] = df['Category'].apply(assign_merchant)

In [21]:
import numpy as np

# reproducibility
rng = np.random.default_rng(seed=42)

accounts = ["Checking Account", "Credit Card", "Savings Account", "Cash Wallet"]

def assign_account(row):
    amount = row['Amount']
    category = row['Category'].lower()
    txn_type = row['Type'].lower()
    
    # Income handling
    if txn_type == "income":
        if amount > 1500:
            return "Savings Account"
        else:
            return rng.choice(["Checking Account", "Savings Account"], p=[0.7, 0.3])
    
    # Large expenses
    if amount > 1500:
        return "Savings Account"
    
    # Category-based
    if category in ["rent", "investment"]:
        return "Savings Account"
    if category in ["utilities", "health & fitness", "other"]:
        return "Checking Account"
    if category in ["shopping", "entertainment", "food & drink"]:
        return rng.choice(["Credit Card", "Checking Account"], p=[0.6, 0.4])
    
    # Small amount edge-case
    if amount < 50:
        return rng.choice(["Cash Wallet", "Checking Account"], p=[0.8, 0.2])
    
    # Default
    return "Checking Account"

df['Account Name'] = df.apply(assign_account, axis=1)

In [22]:
# checking the unique values of Category and Type 
df['Category'].unique()

array(['Food & Drink', 'Utilities', 'Rent', 'Investment', 'Shopping',
       'Other', 'Entertainment', 'Health & Fitness', 'Salary', 'Travel'],
      dtype=object)

In [23]:
df['Type'].value_counts()

Type
Expense    1076
Income      424
Name: count, dtype: int64

Account Name assignment

In [24]:
rng = np.random.default_rng(42)

def assign_account(row):
    amount = row["Amount"]
    cat = row["Category"].lower()
    t = row["Type"].lower()

    if t == "income":
        return "Savings Account" if amount > 1500 else rng.choice(["Checking Account","Savings Account"], p=[0.7,0.3])

    if amount > 1500:
        return "Savings Account"

    if cat in ["rent", "investment"]:
        return "Savings Account"

    if cat in ["utilities", "health & fitness", "other"]:
        return "Checking Account"

    if cat in ["shopping", "entertainment", "food & drink"]:
        return rng.choice(["Credit Card","Checking Account"], p=[0.6,0.4])

    if amount < 50:
        return rng.choice(["Cash Wallet","Checking Account"], p=[0.8,0.2])

    return "Checking Account"

df["Account Name"] = df.apply(assign_account, axis=1)

Validate Required Columns

In [25]:
required = ["TransactionID","UserID","Date","Category","Amount","Type"]
print("[INFO] Missing important fields:")
print(df[required].isna().sum())

[INFO] Missing important fields:
TransactionID    0
UserID           0
Date             0
Category         0
Amount           0
Type             0
dtype: int64


SAVE CLEANED DATASET

In [26]:
import os

# Make sure folder exists
os.makedirs("Tofinal", exist_ok=True)

# Save file into that folder
df.to_csv("Tofinal/Personal_Finance_Dataset.csv", index=False)
print("File saved successfully: Tofinal/Personal_Finance_Dataset.csv")

File saved successfully: Tofinal/Personal_Finance_Dataset.csv
